# Sinhala Tokenizer Extension for Qwen3-4B

Builds `Extended-Sinhala-Qwen3`: the Qwen3-4B tokenizer extended with a learned Sinhala
vocabulary, following the design of the published SinLlama tokenizer
(`polyglots/Extended-Sinhala-LLaMA`, arXiv:2508.09115). The output is what the CPT
notebook consumes as `TOKENIZER_ID` (base model `Qwen/Qwen3-4B-Base`; embeddings are
resized at CPT time — see the final cell).

## How the extension works

Three steps, all in this notebook — **no tiktoken involved**:

1. **Learn a Sinhala vocabulary.** A standalone BPE tokenizer is trained on the Sinhala
   corpus with the Hugging Face `tokenizers` library (Rust backend): `models.BPE` +
   NFC normalizer + `Metaspace` pre-tokenizer. This trained tokenizer is a *throwaway* —
   only the vocabulary it learns matters. `Metaspace` marks word boundaries during
   training, which is what later yields leading-space tokens (`" ලංකාව"`) so that words
   encode as a single token mid-sentence.
2. **Filter the candidates.** The boundary marker `▁` is converted to a literal space and
   only tokens made of Sinhala-block characters (U+0D80–U+0DFF) plus ZWJ (U+200D — needed
   for conjuncts such as `ප්‍ර`) survive, with at most one leading space. BPE rank order
   (≈ corpus frequency) is preserved, so any cap keeps the most useful tokens.
3. **Graft onto Qwen.** `tokenizer.add_tokens(candidates)` appends the survivors to the
   Qwen3 tokenizer as **added tokens** with new contiguous IDs after the base vocabulary.
   At encode time the added-token trie runs *before* Qwen's byte-level BPE: Sinhala spans
   are matched greedily (longest match first), and any text not matched falls through to
   the original byte-level BPE unchanged. Base vocabulary, token IDs, special tokens, and
   the chat template are untouched — a **strict extension**, verified cell-by-cell below.

**Why not tiktoken?** The SinLlama paper describes training its Sinhala vocabulary "using
tiktoken", but tiktoken is an inference-only BPE library — OpenAI has not released its
trainer. What matters is the shipped artifact, not the training tool: inspecting the
published `polyglots/Extended-Sinhala-LLaMA` shows its 11,080 Sinhala tokens stored in the
tokenizer's *added-vocabulary* as raw text strings — exactly the mechanism `add_tokens()`
produces. This notebook reproduces that artifact design on Qwen with maintained,
reproducible tooling. The measured SinLlama design facts mirrored here: tokens appended
contiguously after the base vocab; ~75 % carrying a literal leading space; 707 containing
ZWJ; bare rare codepoints included for coverage.

There are **no hard limits by default**: every Sinhala token the BPE learns (above a
minimal frequency floor) is added. The config knobs can reimpose SinLlama's exact numbers
for a strict replication — their values are noted inline.

Measured baseline this fixes: Qwen3 tokenizes Sinhala at **9.19 tokens/word** (byte-level
fallback); the SinLlama tokenizer reaches **1.60** on the same data.

In [ ]:
%uv pip install -q "transformers>=4.51" "tokenizers>=0.19" "huggingface_hub>=0.23" hf_transfer

In [ ]:
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ---- Base tokenizer ----
# The tokenizer is identical across Qwen3-4B-Base / -4B / -4B-Instruct-2507; Base is named
# here because CPT trains on the Base checkpoint.
BASE_TOKENIZER_ID = "Qwen/Qwen3-4B-Base"

# ---- Sinhala corpus (same one the CPT notebooks use) ----
CORPUS_CANDIDATES = [
    Path("/tmp/sinhala_corpus.txt"),
    Path("./sinhala_data/sinhala_corpus.txt"),
]
DATASET_REPO = "isji/sinhala-corpus"       # fallback download if no local copy
DATASET_FILENAME = "sinhala_corpus.txt"

# ---- Extension knobs (permissive by default: keep everything the corpus supports) ----
# TRAINER_VOCAB_SIZE is the *training* budget of the throwaway BPE, not the number of tokens
# added — it just has to be comfortably larger than the Sinhala vocabulary you expect to
# harvest after filtering. Raise it if the harvest report below shows the filter consuming
# nearly the whole budget.
TRAINER_VOCAB_SIZE = 32_000
# Merges seen fewer than this many times are noise (typos, OCR artifacts); 1 disables.
MIN_FREQUENCY = 2
# None = no cap: every filtered candidate is added. Set 11_080 to mirror SinLlama exactly.
TARGET_NEW_TOKENS = None
# None = no length cap. (For reference, SinLlama's longest added token is 16 chars.)
MAX_TOKEN_CHARS = None

# ---- Tokenizer-training corpus size (independent of the CPT corpus size) ----
# BPE *vocabulary* training converges on a fraction of the data a full continual-pretraining
# run needs — SentencePiece's own trainer has a dedicated `input_sentence_size` flag for
# exactly this reason. This step is CPU/RAM-bound (a GPU box's HBM does nothing for it): the
# trainer builds a word-frequency table over the whole input, and Sinhala's agglutinative
# morphology produces a very large number of unique word forms, so training directly on all
# 10M+ corpus lines can need tens of GB of system RAM and a long wall-clock time. Capping the
# input to a random, seeded subsample keeps memory bounded and is standard practice — it does
# NOT affect the separate CPT notebook, which still trains the model on the full corpus.
# Set to None (or >= the corpus line count) to disable subsampling and use every line.
TOKENIZER_TRAIN_MAX_LINES = 2_000_000
HEARTBEAT_INTERVAL_SECONDS = 15  # live "is it still working" ping during BPE training

# ---- Output ----
OUTPUT_DIR = Path("/tmp/extended-sinhala-qwen3-tokenizer")
PUSH_TO_HUB = False
HF_REPO_ID = "isji/Extended-Sinhala-Qwen3"   # change before enabling PUSH_TO_HUB
HF_REPO_PRIVATE = True

# ---- Fertility evaluation (optional, skipped if absent) ----
TEST_JSONL_CANDIDATES = [
    Path("/tmp/test.jsonl"),
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
]

SEED = 42

SINHALA_BLOCK = r"඀-෿"
ZWJ = "‍"
# One optional leading space, then only Sinhala-block characters and ZWJ.
ALLOWED_TOKEN_RE = re.compile(rf"^ ?[{SINHALA_BLOCK}{ZWJ}]+$")
SINHALA_CHAR_RE = re.compile(rf"[{SINHALA_BLOCK}]")

print("Base tokenizer     :", BASE_TOKENIZER_ID)
print("BPE training budget:", TRAINER_VOCAB_SIZE)
print("Token cap          :", "none (keep all filtered candidates)" if TARGET_NEW_TOKENS is None else TARGET_NEW_TOKENS)
print("Output dir         :", OUTPUT_DIR)

In [ ]:
# Only needed for a private corpus repo or for pushing the result.
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])

In [ ]:
corpus_path = next((p for p in CORPUS_CANDIDATES if p.is_file()), None)
if corpus_path is None:
    from huggingface_hub import hf_hub_download
    print(f"No local corpus found; downloading {DATASET_FILENAME} from {DATASET_REPO} ...")
    corpus_path = Path(
        hf_hub_download(repo_id=DATASET_REPO, filename=DATASET_FILENAME, repo_type="dataset")
    )

size_mb = corpus_path.stat().st_size / 1e6
line_count = 0
char_count = 0
sinhala_char_count = 0
with corpus_path.open("r", encoding="utf-8", errors="replace") as handle:
    for line in handle:
        line_count += 1
        char_count += len(line)
        sinhala_char_count += sum(1 for ch in line if "඀" <= ch <= "෿")

print(f"Corpus            : {corpus_path}")
print(f"Size              : {size_mb:,.1f} MB, {line_count:,} lines, {char_count:,} chars")
print(f"Sinhala char frac : {sinhala_char_count / max(char_count, 1):.2%}")
if sinhala_char_count / max(char_count, 1) < 0.5:
    print("WARNING: corpus is less than half Sinhala — check that this is the right file.")

In [ ]:
# Reservoir-sample the corpus for tokenizer training if it's larger than the configured cap.
# Algorithm R: a single streaming pass, memory bounded by the sample size (not the corpus
# size) — the full corpus is never held in memory at once, so this is safe even at 10M+
# lines. This only affects what THIS notebook trains the vocabulary on; the CPT notebook
# still trains the model itself on the full, unsampled corpus.
import random

random.seed(SEED)

if TOKENIZER_TRAIN_MAX_LINES is None or line_count <= TOKENIZER_TRAIN_MAX_LINES:
    tokenizer_train_path = corpus_path
    print(f"Corpus has {line_count:,} lines <= cap; training on the full file directly.")
else:
    reservoir = []
    with corpus_path.open("r", encoding="utf-8", errors="replace") as handle:
        for index, line in enumerate(handle):
            if index < TOKENIZER_TRAIN_MAX_LINES:
                reservoir.append(line)
            else:
                swap = random.randint(0, index)
                if swap < TOKENIZER_TRAIN_MAX_LINES:
                    reservoir[swap] = line

    tokenizer_train_path = Path("/tmp/tokenizer_train_sample.txt")
    with tokenizer_train_path.open("w", encoding="utf-8", newline="\n") as handle:
        handle.writelines(reservoir)

    sample_size_mb = tokenizer_train_path.stat().st_size / 1e6
    print(
        f"Subsampled {len(reservoir):,} / {line_count:,} lines "
        f"({len(reservoir) / line_count:.1%}) -> {tokenizer_train_path} "
        f"({sample_size_mb:,.1f} MB, was {size_mb:,.1f} MB)"
    )
    del reservoir  # free before training starts

In [ ]:
# Train a whitespace-aware BPE on the (possibly subsampled) Sinhala text. Metaspace
# pre-tokenization marks word-initial pieces, which after conversion become the
# literal-leading-space added tokens the SinLlama design uses. The trained tokenizer itself
# is a throwaway — only its learned vocabulary matters, because tokens are grafted onto Qwen
# as raw-text added tokens.
#
# `BpeTrainer.train()` is one blocking Rust call with no Python-side progress callback;
# `show_progress=True` prints the Rust library's own indicatif bars, which is the only
# per-step signal available but does not reliably render in captured/remote notebook output
# (e.g. Modal) — total silence for several minutes does not by itself mean it is stuck. The
# heartbeat thread below prints elapsed time and this process's peak memory (RSS) every
# HEARTBEAT_INTERVAL_SECONDS while training runs, so "is it alive" always has a real answer;
# it relies on the Rust extension releasing the GIL during the training call (standard for
# long-running PyO3 bindings) — if it doesn't, the heartbeat simply prints nothing until the
# cell finishes, which is a harmless no-op, not a failure.
import threading
import time

from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers

try:  # Linux/macOS (Modal is Linux) — no extra dependency.
    import resource

    def _peak_rss_gb():
        # ru_maxrss is KB on Linux, bytes on macOS; Modal runs Linux, so KB is assumed.
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6
except ImportError:  # Windows fallback for local dry runs; not the production target.
    try:
        import psutil

        _process = psutil.Process()

        def _peak_rss_gb():
            return _process.memory_info().rss / 1e9
    except ImportError:

        def _peak_rss_gb():
            return float("nan")

sin_bpe = Tokenizer(models.BPE(unk_token=None))
sin_bpe.normalizer = normalizers.NFC()
try:
    sin_bpe.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="always")
except TypeError:  # older tokenizers API
    sin_bpe.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁", add_prefix_space=True)

trainer = trainers.BpeTrainer(
    vocab_size=TRAINER_VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=[],
    show_progress=True,
)

train_file_size_mb = tokenizer_train_path.stat().st_size / 1e6
print(f"Training BPE on {tokenizer_train_path} ({train_file_size_mb:,.1f} MB)...")
print("(Rust progress bars print below if the environment renders them; "
      "the heartbeat line below prints regardless, every "
      f"{HEARTBEAT_INTERVAL_SECONDS}s.)")

train_start = time.perf_counter()
training_done = threading.Event()


def _heartbeat():
    while not training_done.wait(HEARTBEAT_INTERVAL_SECONDS):
        elapsed = time.perf_counter() - train_start
        print(f"[heartbeat] still training... elapsed {elapsed:,.0f}s | peak RSS so far: "
              f"{_peak_rss_gb():,.2f} GB", flush=True)


heartbeat_thread = threading.Thread(target=_heartbeat, daemon=True)
heartbeat_thread.start()
try:
    sin_bpe.train([str(tokenizer_train_path)], trainer)
finally:
    training_done.set()
    heartbeat_thread.join(timeout=1)

train_seconds = time.perf_counter() - train_start

trained_vocab = sin_bpe.get_vocab()
mb_per_second = train_file_size_mb / max(train_seconds, 1e-9)
print(f"\nBPE training finished in {train_seconds:,.1f}s "
      f"({train_seconds / 60:,.1f} min) — {mb_per_second:,.2f} MB/s")
print(f"Peak memory (RSS) during this cell: {_peak_rss_gb():,.2f} GB")
print(f"Trained BPE vocabulary: {len(trained_vocab):,} entries")

In [ ]:
# Convert to raw-text candidates and filter to the charset design: optional single leading
# space + Sinhala block + ZWJ only. Rank order (= BPE merge priority, i.e. corpus frequency)
# is preserved, so if a cap is set it keeps the most frequent tokens.
candidates = []
seen = set()
for token, token_id in sorted(trained_vocab.items(), key=lambda kv: kv[1]):
    raw = token.replace("▁", " ")
    if not ALLOWED_TOKEN_RE.match(raw):
        continue
    if MAX_TOKEN_CHARS is not None and len(raw) > MAX_TOKEN_CHARS + 1:  # +1 allows the leading space
        continue
    if raw in seen:
        continue
    seen.add(raw)
    candidates.append(raw)

print(f"Sinhala candidates after filtering: {len(candidates):,} "
      f"(of {len(trained_vocab):,} trained entries)")
if len(candidates) > 0.9 * TRAINER_VOCAB_SIZE:
    print("NOTE: harvest is close to the training budget — consider raising TRAINER_VOCAB_SIZE.")
new_tokens = list(candidates) if TARGET_NEW_TOKENS is None else candidates[:TARGET_NEW_TOKENS]

# Coverage guarantee: every Sinhala codepoint that appears in the corpus gets a bare token
# (SinLlama does this too — its added vocab ends with rare codepoints like ෬ ෭ ෮).
existing = set(new_tokens)
coverage_added = 0
with corpus_path.open("r", encoding="utf-8", errors="replace") as handle:
    corpus_codepoints = set()
    for line in handle:
        corpus_codepoints.update(ch for ch in line if "඀" <= ch <= "෿")
for ch in sorted(corpus_codepoints):
    if ch not in existing:
        new_tokens.append(ch)
        existing.add(ch)
        coverage_added += 1

space_prefixed = sum(1 for t in new_tokens if t.startswith(" "))
zwj_tokens = sum(1 for t in new_tokens if ZWJ in t)
lengths = sorted(len(t) for t in new_tokens)

print(f"Final new-token list : {len(new_tokens):,}")
print(f"  space-prefixed     : {space_prefixed:,} ({space_prefixed / len(new_tokens):.0%})")
print(f"  containing ZWJ     : {zwj_tokens:,}")
print(f"  singleton coverage : {coverage_added}")
print(f"  length min/med/max : {lengths[0]} / {lengths[len(lengths) // 2]} / {lengths[-1]}")
print("  first 15           :", new_tokens[:15])

In [ ]:
from transformers import AutoTokenizer

base_tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER_ID)
extended_tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER_ID)

base_len = len(base_tokenizer)
base_specials = {
    name: getattr(base_tokenizer, name)
    for name in ("bos_token", "eos_token", "pad_token", "unk_token")
}
base_chat_template = extended_tokenizer.chat_template

added_count = extended_tokenizer.add_tokens(new_tokens)
extended_len = len(extended_tokenizer)

print(f"Base vocabulary     : {base_len:,}")
print(f"Requested additions : {len(new_tokens):,}")
print(f"Actually added      : {added_count:,} (duplicates skipped automatically)")
print(f"Extended vocabulary : {extended_len:,}")
print(f"New ID range        : {base_len:,} .. {extended_len - 1:,}")

In [ ]:
# ---- Verification 1: strict extension ----
# Every base token ID must map to the same token string in both tokenizers. This is the
# same guarantee the CPT notebook re-checks before spending GPU time.
def first_token_id_mismatch(left, right, count, chunk_size=8192):
    for start in range(0, count, chunk_size):
        ids = list(range(start, min(start + chunk_size, count)))
        for token_id, l_tok, r_tok in zip(
            ids, left.convert_ids_to_tokens(ids), right.convert_ids_to_tokens(ids)
        ):
            if l_tok != r_tok:
                return token_id, l_tok, r_tok
    return None

mismatch = first_token_id_mismatch(base_tokenizer, extended_tokenizer, base_len)
if mismatch is not None:
    raise RuntimeError(f"NOT a strict extension — first mismatch: {mismatch}")
print(f"Strict extension verified over all {base_len:,} base IDs.")

# ---- Verification 2: special tokens and chat template unchanged ----
for name, value in base_specials.items():
    now = getattr(extended_tokenizer, name)
    if now != value:
        raise RuntimeError(f"{name} changed: {value!r} -> {now!r}")
if extended_tokenizer.chat_template != base_chat_template:
    raise RuntimeError("Chat template changed during extension.")
print("Special tokens and chat template unchanged.")

# ---- Verification 3: English and code tokenization is bit-identical ----
regression_samples = [
    "The king ruled the country for twenty years during the colonial period.",
    "def f(x):\n    return x ** 2  # comment",
    "<|im_start|>user\nHello<|im_end|>",
    "Numbers 1815, 1948 and mixed English-ලංකා text.",
]
for sample in regression_samples[:3]:  # pure English/code must be untouched
    before = base_tokenizer(sample, add_special_tokens=False)["input_ids"]
    after = extended_tokenizer(sample, add_special_tokens=False)["input_ids"]
    if before != after:
        raise RuntimeError(f"Non-Sinhala tokenization changed for: {sample!r}")
print("English/code/special-token tokenization unchanged.")

In [ ]:
# ---- Verification 4: round-trip exactness on Sinhala ----
roundtrip_samples = [
    "මම ලංකාව",
    "ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත.",
    "1818 නොවැම්බර් 26 දින ඉංග්‍රීසීන් විසින් කැප්පෙටිපොළ හා මඩුගල්ලේ දෙදෙනා",
    "රජු ය.\nදෙවන පේළිය සහ ඉංග්‍රීසි words mixed 123.",
]
for sample in roundtrip_samples:
    ids = extended_tokenizer(sample, add_special_tokens=False)["input_ids"]
    decoded = extended_tokenizer.decode(ids, skip_special_tokens=False)
    if decoded != sample:
        raise RuntimeError(f"Round-trip failed:\n  in : {sample!r}\n  out: {decoded!r}")
print("Round-trip encode→decode is exact on Sinhala samples.")

for sample in roundtrip_samples[:2]:
    ids = extended_tokenizer(sample, add_special_tokens=False)["input_ids"]
    print(f"{sample!r}")
    print("   ->", extended_tokenizer.convert_ids_to_tokens(ids))

In [ ]:
# ---- Verification 5: fertility before/after ----
test_path = next((p for p in TEST_JSONL_CANDIDATES if p.is_file()), None)
if test_path is None:
    print("No test.jsonl found — skipping fertility measurement.")
else:
    rows = [json.loads(l) for l in test_path.open(encoding="utf-8") if l.strip()]

    def fertility(tok):
        words = toks = 0
        for row in rows:
            for text in (row["context"], row["question"], row.get("answer") or ""):
                words += len(text.split())
                toks += len(tok(text, add_special_tokens=False)["input_ids"])
        return toks / max(words, 1), toks

    base_fert, base_total = fertility(base_tokenizer)
    ext_fert, ext_total = fertility(extended_tokenizer)
    print(f"Measured on {test_path} ({len(rows)} rows):")
    print(f"  Base Qwen3     : {base_fert:.2f} tokens/word  ({base_total:,} total)")
    print(f"  Extended Qwen3 : {ext_fert:.2f} tokens/word  ({ext_total:,} total)")
    print(f"  Reduction      : {1 - ext_total / base_total:.1%}")
    print("  (References: SinLlama tokenizer 1.60 tok/word; base Qwen3 9.19 on this data.)")

    for word in ["රජු", "ලංකාව", "ඉතිහාසය", "ප්‍රශ්නය", "ඉංග්‍රීසීන්"]:
        b = len(base_tokenizer(word, add_special_tokens=False)["input_ids"])
        e = len(extended_tokenizer(word, add_special_tokens=False)["input_ids"])
        print(f"  {word!r}: {b} -> {e} tokens")

In [ ]:
# ---- Save and reload-verify ----
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
extended_tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

reloaded = AutoTokenizer.from_pretrained(OUTPUT_DIR)
if len(reloaded) != extended_len:
    raise RuntimeError("Reloaded tokenizer has a different size.")
mismatch = first_token_id_mismatch(base_tokenizer, reloaded, base_len)
if mismatch is not None:
    raise RuntimeError(f"Reloaded tokenizer lost strict extension: {mismatch}")
sample = "ප්‍රශ්නයට පිළිතුරු"
if reloaded(sample, add_special_tokens=False)["input_ids"] != extended_tokenizer(
    sample, add_special_tokens=False
)["input_ids"]:
    raise RuntimeError("Reloaded tokenizer encodes differently.")
print("Reload verification passed — the saved tokenizer is self-contained and correct.")

In [ ]:
# ---- Embedding-resize arithmetic for the CPT step (informational) ----
from transformers import AutoConfig

config = AutoConfig.from_pretrained(BASE_TOKENIZER_ID)
padded_vocab = config.vocab_size            # 151,936: embedding rows already allocated
hidden = config.hidden_size
tied = getattr(config, "tie_word_embeddings", False)

new_rows_needed = extended_len - padded_vocab
new_params = extended_len * hidden - padded_vocab * hidden
matrices = 1 if tied else 2

print(f"Model config        : hidden={hidden}, padded vocab={padded_vocab:,}, tied={tied}")
print(f"Extended tokenizer  : {extended_len:,} tokens")
print(f"Embedding rows to add beyond existing padding: {max(new_rows_needed, 0):,}")
print(f"New parameters      : {max(new_params, 0) * matrices / 1e6:,.1f} M "
      f"({matrices} matrix/matrices — tied embeddings share one)")
print()
print("At CPT time:  model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=64)")
print("New rows are randomly initialized; CPT with trainable embeddings")
print('(modules_to_save=["embed_tokens"] under LoRA) is what makes them useful.')

In [ ]:
# ---- Optional: push to the Hub ----
if PUSH_TO_HUB:
    extended_tokenizer.push_to_hub(
        HF_REPO_ID,
        private=HF_REPO_PRIVATE,
        commit_message=(
            f"Sinhala-extended Qwen3 tokenizer: {base_len:,} base + {added_count:,} added "
            f"= {extended_len:,} tokens (SinLlama-recipe extension)"
        ),
    )
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")
else:
    print("PUSH_TO_HUB is False — not uploading. Set PUSH_TO_HUB = True and rerun this cell.")

## Next step: CPT on `Qwen/Qwen3-4B-Base`

Point the CPT notebook (`llama-scripts/sinllama_4b_cpt_b200.ipynb`) at this tokenizer with
these changes:

1. `MODEL_ID = "Qwen/Qwen3-4B-Base"` — **Base, not Instruct-2507**: continual pre-training
   on raw text catastrophically forgets instruction tuning; the QA fine-tune afterwards is
   the instruction-recovery step (same as the Llama pipeline).
2. `TOKENIZER_ID = "<this repo or local path>"`.
3. The architecture guard `model_type != "llama"` must accept `"qwen3"`.
4. Qwen3 has no BOS token (`bos_token_id=None`) — the bos/eos equality check passes
   because both sides are None, but any code that *prepends* `bos_token_id` must be
   conditional.
5. After loading: `model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=64)`.
   With `tie_word_embeddings=True` (Qwen3-4B) there is a single shared matrix — include
   `embed_tokens` in `modules_to_save` for LoRA CPT so the new rows train.
6. Keep the strict-extension verification cell — it reruns here against Qwen and protects
   the GPU budget from a mismatched tokenizer upload.